In [13]:
# ============================================================
#  CheXpert Fine-tuning  |  Starting from NIH pretrained model
#  Strategy:
#    1. Load best_model_v3_8labels.pth (trained on NIH)
#    2. Fine-tune on CheXpert (224k images, overlapping labels)
#    3. Save new checkpoint → use as starting point for NIH again
#
#  GPU note: Each epoch ~90-100 min on T4
#            Run with epochs=5 first to validate, then epochs=15
# ============================================================

import os, time, warnings, copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import OneCycleLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

from torchvision import transforms
from PIL import Image
import timm

warnings.filterwarnings("ignore")


# ========================= CONFIG =========================
class CFG:
    # ── Paths ─────────────────────────────────────────────
    chexpert_csv  = "/kaggle/input/datasets/ashery/chexpert/train.csv"
    chexpert_root = "/kaggle/input/datasets/ashery/chexpert/"

    # Your existing NIH trained model — load this as starting weights
    nih_model_path = "/kaggle/input/models/bhurgundy/nih-model-0-43f1/pytorch/default/1/best_model_v3_8labels (1).pth"

    # Where to save the CheXpert fine-tuned model
    save_path      = "chexpert_finetuned.pth"

    # ── Model ─────────────────────────────────────────────
    model_name  = "swin_base_patch4_window7_224"
    img_size    = 224
    drop_rate   = 0.2

    # ── Labels ────────────────────────────────────────────
    # Only use labels that exist in BOTH NIH and CheXpert
    # CheXpert uncertainty (-1) treated as negative (0)
    # "U-Zeros" strategy — most common approach in literature
    CHEXPERT_LABELS = [
        'Atelectasis',      # exact match
        'Cardiomegaly',     # exact match
        'Pleural Effusion', # NIH calls it "Effusion"
        'Pneumothorax',     # exact match
        'Lung Opacity',     # NIH closest = "Infiltration"
        'Lung Lesion',      # NIH closest = "Mass" / "Nodule"
    ]
    num_classes = 6         # 6 overlapping labels

    # ── Training ──────────────────────────────────────────
    epochs       = 15        # change to 3 if testing first
    batch_size   = 32
    accum_steps  = 2
    num_workers  = 4
    seed         = 42

    # ── Lower LR since we're fine-tuning an already good model ──
    base_lr      = 5e-5     # was 2e-4 for NIH from scratch
    llrd_decay   = 0.75
    weight_decay = 0.05

    focal_gamma     = 2.0
    pos_weight_clip = 20.0

    use_weighted_sampler = True
    mixup_alpha = 0.4
    mixup_prob  = 0.5

    use_ema   = True
    ema_decay = 0.999

    threshold_search = np.arange(0.10, 0.90, 0.01)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG.seed)


# ========================= FOCAL LOSS =========================
class FocalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none'
        )
        prob  = torch.sigmoid(logits)
        p_t   = prob * targets + (1 - prob) * (1 - targets)
        return ((1 - p_t) ** self.gamma * bce).mean()


# ========================= DATASET =========================
class CheXpertDataset(Dataset):
    """
    CheXpert dataset loader.
    Handles:
      - Uncertainty labels (-1) → treated as 0 (U-Zeros strategy)
      - Missing labels (NaN)    → treated as 0
      - Frontal + Lateral views (keep both)
    """
    def __init__(self, dataframe, root, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.root      = root
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
    
    # Strip the CheXpert-v1.0-small/ prefix since it doesn't exist on Kaggle
        clean_path = row['Path'].replace("CheXpert-v1.0-small/", "")
        img_path   = os.path.join(self.root, clean_path)

        img = Image.open(img_path).convert("RGB")
        lbl = torch.tensor(
            row[CFG.CHEXPERT_LABELS].values.astype(np.float32)
        )
        if self.transform: img = self.transform(img)
        return img, lbl


# ========================= TRANSFORMS =========================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(CFG.img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# ========================= DATA PREP =========================
def prepare_chexpert():
    df = pd.read_csv(CFG.chexpert_csv)

    print(f"Total CheXpert images: {len(df):,}")
    print(f"Using labels: {CFG.CHEXPERT_LABELS}")

    # Keep only the label columns we need + Path
    df = df[['Path'] + CFG.CHEXPERT_LABELS].copy()

    # U-Zeros: replace -1 (uncertain) and NaN with 0
    df[CFG.CHEXPERT_LABELS] = df[CFG.CHEXPERT_LABELS].fillna(0)
    df[CFG.CHEXPERT_LABELS] = df[CFG.CHEXPERT_LABELS].replace(-1, 0)
    df[CFG.CHEXPERT_LABELS] = df[CFG.CHEXPERT_LABELS].clip(0, 1).astype(np.float32)

    # Split
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=CFG.seed)

    print(f"\nTrain: {len(train_df):,}  |  Val: {len(val_df):,}")
    print("\nClass counts in train:")
    counts = train_df[CFG.CHEXPERT_LABELS].sum().sort_values()
    for lbl, cnt in counts.items():
        bar = "█" * int(cnt / counts.max() * 25)
        print(f"  {lbl:<22} {int(cnt):>6}  {bar}")

    return train_df, val_df


# ========================= CLASS WEIGHTS =========================
def compute_pos_weights(train_df):
    labels_np    = train_df[CFG.CHEXPERT_LABELS].values
    class_counts = labels_np.sum(axis=0) + 1e-6
    pos_weight   = np.clip((len(labels_np) - class_counts) / class_counts,
                           1.0, CFG.pos_weight_clip)
    return torch.tensor(pos_weight, dtype=torch.float32).to(device)


# ========================= WEIGHTED SAMPLER =========================
def make_weighted_sampler(train_df):
    labels_np     = train_df[CFG.CHEXPERT_LABELS].values
    class_counts  = labels_np.sum(axis=0) + 1e-6
    class_weights = 1.0 / class_counts
    sample_weights = np.zeros(len(train_df))
    for i in range(len(train_df)):
        pos_idx = np.where(labels_np[i] == 1)[0]
        sample_weights[i] = (class_weights[pos_idx].max()
                             if len(pos_idx) > 0
                             else class_weights.min() * 0.5)
    return WeightedRandomSampler(
        torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
    )


# ========================= MODEL =========================
class ChestViT(nn.Module):
    def __init__(self, model_name, num_classes, drop_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0, global_pool='avg'
        )
        feat_dim  = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def load_nih_model_as_backbone(nih_path, new_num_classes):
    """
    Load NIH model, keep backbone weights, replace head for CheXpert labels.
    This is transfer learning FROM NIH TO CheXpert.
    """
    print(f"Loading NIH pretrained backbone from: {nih_path}")

    # Load NIH checkpoint
    nih_ckpt = torch.load(nih_path, map_location=device, weights_only=False)

    # Build NIH model (8 classes) to extract backbone weights
    nih_model = ChestViT("swin_base_patch4_window7_224", 8, 0.2)
    nih_model.load_state_dict(nih_ckpt['state_dict'])

    # Build new model for CheXpert (6 classes)
    new_model = ChestViT(CFG.model_name, new_num_classes, CFG.drop_rate)

    # Copy backbone weights (everything except head)
    new_model.backbone.load_state_dict(nih_model.backbone.state_dict())

    # Head is randomly initialized for new classes — that's correct
    print(f"✅ Backbone loaded from NIH model (F1={nih_ckpt['f1_macro']:.4f})")
    print(f"   New head initialized for {new_num_classes} CheXpert classes")

    return new_model.to(device)


# ========================= EMA =========================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

    def get_model(self): return self.shadow


# ========================= LLRD OPTIMIZER =========================
def build_llrd_optimizer(model, base_lr, decay, weight_decay):
    param_groups = [{'params': list(model.head.parameters()),
                     'lr': base_lr, 'weight_decay': weight_decay}]
    for i, layer in enumerate(reversed(list(model.backbone.children()))):
        param_groups.append({
            'params': list(layer.parameters()),
            'lr': base_lr * (decay ** (i + 1)),
            'weight_decay': weight_decay
        })
    return torch.optim.AdamW(param_groups)


# ========================= MIXUP =========================
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)


# ========================= TRAIN =========================
def train_one_epoch(model, ema, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, (images, labels_batch) in enumerate(loader):
        images       = images.to(device)
        labels_batch = labels_batch.to(device)

        with autocast():
            if np.random.rand() < CFG.mixup_prob:
                mx, ya, yb, lam = mixup_data(images, labels_batch, CFG.mixup_alpha)
                loss = mixup_loss(criterion, model(mx), ya, yb, lam)
            else:
                loss = criterion(model(images), labels_batch)
            loss = loss / CFG.accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % CFG.accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            if CFG.use_ema: ema.update(model)

        total_loss += loss.item() * CFG.accum_steps

    return total_loss / len(loader)


# ========================= EVALUATE =========================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    for images, labels_batch in loader:
        images = images.to(device)
        probs  = torch.sigmoid(model(images)).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(labels_batch.numpy())
    return np.vstack(all_preds), np.vstack(all_targets)


# ========================= THRESHOLDS =========================
def optimize_thresholds(preds, targets):
    best = []
    for i in range(targets.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in CFG.threshold_search:
            f1 = f1_score(targets[:, i], (preds[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        best.append(best_t)
    return best


# ========================= LOG =========================
def log_metrics(preds, targets, thresholds, epoch, loss, tag=""):
    bp = np.zeros_like(preds)
    for i, t in enumerate(thresholds):
        bp[:, i] = (preds[:, i] > t).astype(int)

    f1_mac = f1_score(targets, bp, average='macro',    zero_division=0)
    f1_mic = f1_score(targets, bp, average='micro',    zero_division=0)
    pcf1   = f1_score(targets, bp, average=None,       zero_division=0)

    aucs = [roc_auc_score(targets[:, i], preds[:, i])
            for i in range(targets.shape[1])
            if len(np.unique(targets[:, i])) > 1]
    mean_auc = np.mean(aucs) if aucs else 0.0

    print(f"\n{'='*60}")
    print(f"  Epoch {epoch+1:>2}  {tag}  |  Loss: {loss:.4f}")
    print(f"  F1 Macro: {f1_mac:.4f}  |  F1 Micro: {f1_mic:.4f}  |  AUC: {mean_auc:.4f}")
    for lbl, f1, t in zip(CFG.CHEXPERT_LABELS, pcf1, thresholds):
        bar  = "█" * int(f1 * 20)
        flag = " ← LOW" if f1 < 0.2 else ""
        print(f"    {lbl:<22} {f1:.3f}  t={t:.2f}  {bar}{flag}")
    print(f"{'='*60}")
    return f1_mac, mean_auc


# ========================= MAIN =========================
def main():
    train_df, val_df = prepare_chexpert()

    train_ds = CheXpertDataset(train_df, CFG.chexpert_root, train_transform)
    val_ds   = CheXpertDataset(val_df,   CFG.chexpert_root, val_transform)

    sampler = make_weighted_sampler(train_df) if CFG.use_weighted_sampler else None
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size,
                              sampler=sampler, shuffle=(sampler is None),
                              num_workers=CFG.num_workers, pin_memory=True,
                              drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size,
                              shuffle=False, num_workers=CFG.num_workers,
                              pin_memory=True)

    # Load NIH backbone, replace head for CheXpert
    model = load_nih_model_as_backbone(CFG.nih_model_path, CFG.num_classes)
    ema   = EMA(model, CFG.ema_decay) if CFG.use_ema else None

    pos_weight = compute_pos_weights(train_df)
    criterion  = FocalBCELoss(pos_weight=pos_weight, gamma=CFG.focal_gamma)

    # Lower LR since backbone is already good from NIH training
    optimizer = build_llrd_optimizer(model, CFG.base_lr,
                                     CFG.llrd_decay, CFG.weight_decay)

    steps_per_epoch = len(train_loader) // CFG.accum_steps
    scheduler = OneCycleLR(
        optimizer,
        max_lr          = [pg['lr'] for pg in optimizer.param_groups],
        steps_per_epoch = steps_per_epoch,
        epochs          = CFG.epochs,
        pct_start       = 0.15,     # longer warmup
        anneal_strategy = 'cos',
        div_factor      = 25,
        final_div_factor= 1e4
    )

    scaler     = GradScaler()
    best_f1    = 0.0
    best_auc   = 0.0
    thresholds = [0.5] * CFG.num_classes

    for epoch in range(CFG.epochs):
        t0   = time.time()
        loss = train_one_epoch(model, ema, train_loader, criterion,
                               optimizer, scheduler, scaler)

        preds, targets = evaluate(model, val_loader)
        thresholds     = optimize_thresholds(preds, targets)
        f1_mac, auc    = log_metrics(preds, targets, thresholds, epoch, loss, "[raw]")

        if CFG.use_ema:
            ep, et    = evaluate(ema.get_model(), val_loader)
            eth       = optimize_thresholds(ep, et)
            ef1, eauc = log_metrics(ep, et, eth, epoch, loss, "[EMA]")
            if ef1 > f1_mac:
                f1_mac, auc, thresholds = ef1, eauc, eth

        print(f"  Time: {(time.time()-t0)/60:.1f} min")

        if f1_mac > best_f1:
            best_f1, best_auc = f1_mac, auc
            save_m = ema.get_model() if CFG.use_ema else model
            torch.save({
                'epoch':           epoch,
                'state_dict':      save_m.state_dict(),
                'thresholds':      thresholds,
                'f1_macro':        best_f1,
                'mean_auc':        best_auc,
                'chexpert_labels': CFG.CHEXPERT_LABELS,
                'model_name':      CFG.model_name,
            }, CFG.save_path)
            print(f"  ✅ Saved  F1={best_f1:.4f}  AUC={best_auc:.4f}")

    print(f"\n🏁 CheXpert training done.")
    print(f"   Best F1={best_f1:.4f}  AUC={best_auc:.4f}")
    print(f"\n   Next step: use {CFG.save_path} as starting weights for NIH fine-tuning")
    print(f"   Expected NIH F1 after fine-tuning back: 0.50–0.58")


if __name__ == "__main__":
    main()

Device: cuda
Total CheXpert images: 223,414
Using labels: ['Atelectasis', 'Cardiomegaly', 'Pleural Effusion', 'Pneumothorax', 'Lung Opacity', 'Lung Lesion']

Train: 201,072  |  Val: 22,342

Class counts in train:
  Lung Lesion              8275  ██
  Pneumothorax            17469  ████
  Cardiomegaly            24300  ██████
  Atelectasis             29972  ███████
  Pleural Effusion        77598  ████████████████████
  Lung Opacity            95007  █████████████████████████
Loading NIH pretrained backbone from: /kaggle/input/models/bhurgundy/nih-model-0-43f1/pytorch/default/1/best_model_v3_8labels (1).pth
✅ Backbone loaded from NIH model (F1=0.4070)
   New head initialized for 6 CheXpert classes

  Epoch  1  [raw]  |  Loss: 0.3166
  F1 Macro: 0.4816  |  F1 Micro: 0.5893  |  AUC: 0.7759
    Atelectasis            0.339  t=0.58  ██████
    Cardiomegaly           0.475  t=0.63  █████████
    Pleural Effusion       0.728  t=0.49  ██████████████
    Pneumothorax           0.444  t=0.64  █

KeyboardInterrupt: 